In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import gzip

In [ ]:

excluded_samples = ['ST002-1D_LUNG-pacbio-uwsc-group1']

In [ ]:
## read in mitoscope qc files
df_pb = pd.read_csv("../benchmark/pacbio/output/qc_summary.tsv", sep='\t')
df_pb[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = df_pb['Sample'].str.split('-', expand=True)
df_pb['Age'] = np.where(df_pb['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

df_ont = pd.read_csv("../benchmark/ont/output/qc_summary.tsv", sep='\t')
df_ont[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = df_ont['Sample'].str.split('-', expand=True)
df_ont['Age'] = np.where(df_ont['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

df = pd.concat([df_pb, df_ont]).reset_index()
df['Donor_Tissue'] = df['Donor'] + "-" + df['Tissue']
df = df.sort_values(['Tissue', 'Donor', 'Center'])
df = df[~(df['Sample'].isin(excluded_samples))]
df['Tissue'] = df['Tissue'].str.split('_', expand=True)[1].str.capitalize()

df

In [ ]:
## generate combined df with deletions called across all PB and ONT samples
comb_del_df = pd.DataFrame()

for sample in df_pb['Sample']:
    file_path = f'../benchmark/pacbio/output/{sample}/variants/baldur/{sample}.mt.baldur_del.txt'
    del_df = pd.read_csv(file_path, sep='\t')
    del_df['Sample'] = sample
    comb_del_df = pd.concat([comb_del_df, del_df])

for sample in df_ont['Sample']:
    file_path = f'../benchmark/ont/output/{sample}/variants/baldur/{sample}.mt.baldur_del.txt'
    del_df = pd.read_csv(file_path, sep='\t')
    del_df['Sample'] = sample
    comb_del_df = pd.concat([comb_del_df, del_df])

comb_del_df = comb_del_df[comb_del_df['Type'] != 'Wildtype']
comb_del_df[['Start', 'End', 'Size', 'Fwd', 'Rev']] = comb_del_df[['Start', 'End', 'Size', 'Fwd', 'Rev']].astype(int)
comb_del_df['read_support'] = comb_del_df ['Fwd'] + comb_del_df ['Rev']
comb_del_df['Size'] = np.where(comb_del_df['Size'] < 0, -(comb_del_df['Size']), comb_del_df['Size'])

comb_del_df = pd.merge(comb_del_df, df[['Sample', 'Mito_Read_Count']], on = 'Sample')
comb_del_df['per_mtdna'] = comb_del_df['read_support'] / comb_del_df['Mito_Read_Count']
comb_del_df[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = comb_del_df['Sample'].str.split('-', expand=True)
comb_del_df['Donor_Tissue'] = comb_del_df['Donor'] + "_" + comb_del_df['Tissue']
comb_del_df['Age'] = np.where(comb_del_df['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

comb_del_df = comb_del_df[comb_del_df['Size'] > 45]
comb_del_df = comb_del_df.sort_values(by=['read_support','Donor', 'Tissue', 'Center'])
comb_del_df['Tissue'] = comb_del_df['Tissue'].str.split('_', expand=True)[1].str.capitalize()
comb_del_df = comb_del_df[~(comb_del_df['Sample'].isin(excluded_samples))]

comb_del_df

In [ ]:
## output tsv file with unique deletion coordinates for microhomology analysis
#comb_del_df.sort_values(['Start', 'End'], ascending=True)[['Start', 'End']].drop_duplicates().to_csv('deletion_coordinates_nodups.tsv', sep='\t', index=False)

In [ ]:
## plot deletion coverage plots 
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

mt_len = 16569

comb_del_df_sorted = comb_del_df.copy().sort_values('Start')
comb_del_df_sorted["y"] = comb_del_df_sorted.groupby("Tissue").cumcount()

for sample, sdf in comb_del_df_sorted.groupby("Tissue"):

    fig, (ax_cov, ax_bar) = plt.subplots(
        2, 1,
        figsize=(12, 4),
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1.5]}
    )

    coverage = np.zeros(mt_len, dtype=int)

    for _, row in sdf.iterrows():
        coverage[row["Start"]:row["End"]] += row["read_support"]

    ax_cov.plot(coverage, color="black", linewidth=1)
    ax_cov.fill_between(range(mt_len), coverage, alpha=0.5)
    ax_cov.set_ylim(0, max(coverage) + max(coverage)*0.1)
    ax_cov.set_ylabel("")
    ax_cov.set_title(sample)
    ax_cov.set_yticks([0, max(coverage)])

    for _, row in sdf.iterrows():
        ax_bar.hlines(
            y=row["y"],
            xmin=row["Start"],
            xmax=row["End"],
            linewidth=1,
            alpha=0.7,
            color="steelblue"
        )

    ax_bar.set_xlim(0, mt_len)
    ax_bar.set_ylim(-0.5, sdf["y"].max() + 0.5)
    ax_bar.set_xlabel("")
    ax_bar.set_yticks([])
    ax_bar.set_ylabel("")

    plt.tight_layout()
    plt.savefig(f"plots/fig5-deletion_spanning_region_{sample}.pdf", dpi=300)
    plt.show()


In [ ]:
## get deletion frequencies per general region (control region, major arc, minor arc) or specific deletion (m.298_347del50, common deletion)
crd = comb_del_df[
    (comb_del_df['Start'].isin(range(0,576))) | (comb_del_df['Start'].isin(range(16024,16569))) | (comb_del_df['End'].isin(range(0,576))) | (comb_del_df['End'].isin(range(16024,16569))) 
      ].groupby('Sample')['read_support'].sum().reset_index()
crd2 = comb_del_df[
    (comb_del_df['Start'].isin(range(287,307))) | (comb_del_df['Start'].isin(range(336,356))) | (comb_del_df['End'].isin(range(287,307))) | (comb_del_df['End'].isin(range(336,356))) 
      ].groupby('Sample')['read_support'].sum().reset_index()
common_del = comb_del_df[((comb_del_df['Start'].isin(range(8460,8480))) & (comb_del_df['End'].isin(range(13430,13455)))) | ((comb_del_df['End'].isin(range(8460,8480))) & (comb_del_df['Start'].isin(range(13430,13455))))].groupby('Sample')['read_support'].sum().reset_index()
major_arc_del = comb_del_df[(comb_del_df['Start'].isin(range(5576,15976))) & (comb_del_df['End'].isin(range(5576,15976)))].groupby('Sample')['read_support'].sum().reset_index()
minor_arc_del = comb_del_df[(comb_del_df['Start'].isin(range(0,5576))) & (comb_del_df['End'].isin(range(0,5576)))].groupby('Sample')['read_support'].sum().reset_index()

crd = crd.set_index(['Sample']).reindex(comb_del_df['Sample'].unique()).fillna(0).reset_index()
crd[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = crd['Sample'].str.split('-', expand=True)
crd['Donor_Tissue'] = crd['Donor'] + "-" + crd['Tissue']
crd = pd.merge(crd, df[['Sample', 'Mito_Read_Count']], on = 'Sample')
crd['per_mtdna'] = crd['read_support'] / crd['Mito_Read_Count']
crd

crd2 = crd2.set_index(['Sample']).reindex(comb_del_df['Sample'].unique()).fillna(0).reset_index()
crd2[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = crd2['Sample'].str.split('-', expand=True)
crd2['Donor_Tissue'] = crd2['Donor'] + "-" + crd2['Tissue']
crd2 = pd.merge(crd2, df[['Sample', 'Mito_Read_Count']], on = 'Sample')
crd2['per_mtdna'] = crd2['read_support'] / crd2['Mito_Read_Count']
crd2

major_arc_del = major_arc_del.set_index(['Sample']).reindex(comb_del_df['Sample'].unique()).fillna(0).reset_index()
major_arc_del[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = major_arc_del['Sample'].str.split('-', expand=True)
major_arc_del['Donor_Tissue'] = major_arc_del['Donor'] + "-" + major_arc_del['Tissue']
major_arc_del = pd.merge(major_arc_del, df[['Sample', 'Mito_Read_Count']], on = 'Sample')
major_arc_del['per_mtdna'] = major_arc_del['read_support'] / major_arc_del['Mito_Read_Count']
major_arc_del

minor_arc_del = minor_arc_del.set_index(['Sample']).reindex(comb_del_df['Sample'].unique()).fillna(0).reset_index()
minor_arc_del[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = minor_arc_del['Sample'].str.split('-', expand=True)
minor_arc_del['Donor_Tissue'] = minor_arc_del['Donor'] + "-" + minor_arc_del['Tissue']
minor_arc_del = pd.merge(minor_arc_del, df[['Sample', 'Mito_Read_Count']], on = 'Sample')
minor_arc_del['per_mtdna'] = minor_arc_del['read_support'] / minor_arc_del['Mito_Read_Count']
minor_arc_del

common_del = common_del.set_index(['Sample']).reindex(comb_del_df['Sample'].unique()).fillna(0).reset_index()
common_del[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = common_del['Sample'].str.split('-', expand=True)
common_del['Donor_Tissue'] = common_del['Donor'] + "-" + common_del['Tissue']
common_del = pd.merge(common_del, df[['Sample', 'Mito_Read_Count']], on = 'Sample')
common_del['per_mtdna'] = common_del['read_support'] / common_del['Mito_Read_Count']
common_del


In [ ]:
#common_del.to_csv(f"tables/fig5a-common_del_freqs.csv", index=False)
#major_arc_del.to_csv(f"tables/fig5a-major_arc_freqs.csv", index=False)
#crd.to_csv(f"tables/fig5a-crd_freqs.csv", index=False)

In [ ]:
crd2.drop_duplicates().sort_values('read_support').groupby('Donor_Tissue')['per_mtdna'].mean()

In [ ]:
## generate deletion frequency plots
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

sns.color_palette("tab10")[0]

color_map = {
    'bcm': sns.color_palette("tab10", 5)[0],
    'broad': sns.color_palette("tab10", 5)[1],
    'nygc': sns.color_palette("tab10", 5)[2],
    'uwsc': sns.color_palette("tab10", 5)[3],
    'washu': sns.color_palette("tab10", 5)[4]
}

for plot_name, plot_type in zip(['Control Region Deletions', 'Common Deletion', 'Major Arc Deletions', 'minor_arc' ], [crd, common_del, major_arc_del, minor_arc_del]):

    fig, ax = plt.subplots(layout='constrained', figsize=(6, 4))

    sns.stripplot(
        data=plot_type[plot_type['Seq_Tech'] == 'pacbio'],
        x="Donor_Tissue",
        y="per_mtdna",
        hue="Center",
    #    style="Seq_Tech",
        s=7,
        legend=False,
        alpha=0.9,
        marker="o",
        palette=color_map,
        ax=ax
        )
    
    sns.stripplot(
        data=plot_type[plot_type['Seq_Tech'] == 'ont'],
        x="Donor_Tissue",
        y="per_mtdna",
        hue="Center",
    #    style="Seq_Tech",
        s=7,
        legend=False,
        alpha=0.9,
        marker="X",
        palette=color_map,
        ax=ax
        )
    

    sns.boxplot(showmeans=True,
                meanline=True,
                meanprops={'color': 'k', 'ls': '-', 'lw': 2},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=10,
                x="Donor_Tissue",
                order=['ST001-1A_LIVER', 'ST001-1D_LUNG', 'ST002-1D_LUNG', 'ST002-1G_COLON', 'ST003-1Q_BRAIN', 'ST004-1Q_BRAIN'],
                y="per_mtdna",
                data=plot_type,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax=ax)
    #plt.xticks(rotation=90)
    #plt.ylabel("Frequency")
    #plt.xlabel("")

    # Get original tick labels
    ticks = ax.get_xticklabels()
    labels = [t.get_text() for t in ticks]
    donors  = [l.split("-")[0] for l in labels]  
    tissues = ['\n\n' + l.split("-")[1].split("_")[1] for l in labels]

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(donors, rotation=0)
    ax.set_ylabel("Frequency")
    ax.set_xlabel("")
    ax.set_title(plot_name)


    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
    sec.tick_params('x', length=0)

    for tick in sec.get_xticklabels():
        tick.set_fontweight("bold")
        #tick.set_fontsize(14)

    midpoints = [0.5,2.5,3.5]
    for x in midpoints:
        ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

   # plt.ylim(0,0.025)
    print(labels)

    import matplotlib.ticker as mtick
    formatter = mtick.FormatStrFormatter('%.1e') # '%.2e' formats to scientific notation with 2 decimal places
    ax.yaxis.set_major_formatter(formatter)

    plt.savefig(f"plots/fig5-deletion_freq_{plot_name}.pdf", dpi=300)
    plt.show()




In [ ]:
## generate legend for del freq plots
fig, ax = plt.subplots(figsize=(4, 4))

sns.stripplot(
        data=crd,
        x="Donor_Tissue",
        y="per_mtdna",
        hue="Center",
    #    style="Seq_Tech",
        s=7,
        legend=True,
        alpha=0.9,
        marker="o",
        palette=color_map,
        ax=ax
        )
    
sns.stripplot(
        data=crd,
        x="Donor_Tissue",
        y="per_mtdna",
        hue="Center",
    #    style="Seq_Tech",
        s=7,
        legend=True,
        alpha=0.9,
        marker="X",
        palette=color_map,
        ax=ax
        )

legend = ax.get_legend()

handles = legend.legend_handles if hasattr(legend, "legend_handles") else legend.legendHandles
labels = [t.get_text() if t.get_text() in ['Center', 'Seq_Tech']
           else 'PacBio' if t.get_text() == 'pacbio'
           else 'ONT' if  t.get_text() == 'ont'
           else t.get_text().upper() 
           for t in legend.get_texts() ]
title = legend.get_title().get_text()

# Clear axes AFTER grabbing legend info
ax.clear()
ax.set_axis_off()

leg = ax.legend(
    handles,
    labels,
    title=title,
    loc="center",
)

plt.savefig(
    "plots/fig5-deletion_freq_legend_center.pdf",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
## get overall deletion counts
counts = comb_del_df.groupby('Sample').size().reset_index(name="Del_Count")
counts = pd.merge(counts, df[['Sample', 'Mito_Read_Count']], on='Sample')
counts['norm_count'] = counts['Del_Count'] / counts['Mito_Read_Count']
counts[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = counts['Sample'].str.split('-', expand=True)
counts = counts.sort_values(['Donor', 'Tissue', 'Center'])
counts['Age'] = np.where(counts['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')
counts['Donor_Tissue'] = counts['Donor'] + "_" + counts['Tissue']

counts

In [ ]:
## plot overall deletion frequencies by tissue
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

plt.figure(figsize=(7,5))
g = sns.boxplot(
    data=counts,
    hue="Tissue",
    y="norm_count",
    x="Seq_Tech",
    widths=0.1
)
plt.legend(bbox_to_anchor=(1, 1), loc='best')
plt.ylabel("Proportion of reads with deletions")
plt.xlabel("")
plt.tight_layout()
#plt.yscale('log')
plt.show()

In [ ]:
## read in blast results comparing surrounding regions of deletion breakpoints (for observed and control simulated ones)
hom = pd.read_csv('./homology_analysis/del_flank_all.tsv', sep='\t', skiprows=1, header=None, keep_default_na=False)
hom.columns = ['id_l', 'id_r', 'length', ]
hom['group'] = 'Observed'

hom_control = pd.read_csv('./homology_analysis/del_flanks_control_all.tsv', sep='\t', skiprows=1, header=None, keep_default_na=False)
hom_control.columns = ['id_l', 'id_r', 'length']
hom_control['group'] = 'Expected'

hom_df = pd.concat([hom, hom_control])
hom_df['length'] = np.where(hom_df['length'] == 'NA', '0', hom_df['length'])
hom_df['length'] = hom_df['length'].astype(int)

conditions = [
    (hom_df['length'] == 0),
    (hom_df['length'] >= 4) & (hom_df['length'] < 7),
    (hom_df['length'] >= 7) & (hom_df['length'] < 10),
    (hom_df['length'] >= 10) & (hom_df['length'] < 13),
    (hom_df['length'] >= 13),
]
values = ['0-3', '4-6', '7-9', '10-12', '13+']
hom_df['length_group'] = np.select(conditions, values, default='-')

prop_df = hom_df.groupby('group')['length_group'].value_counts(normalize=True).rename('proportion').reset_index()
prop_df 

In [ ]:
## plot breakpoint microhomology distributions for obs and exp
sns.set_theme(style="ticks", context="talk", font_scale=0.8)
plt.figure(figsize=(9,4))

sns.barplot(data=prop_df.sort_values("length_group"), 
            x="length_group", 
            order=['0-3', '4-6', '7-9', '10-12', '13+'],
                   y='proportion', 
                   hue='group')
plt.yscale('log')
plt.xlabel('Deletion breakpoint microhomology (bp)')
plt.ylabel('Proportion of deletions')
plt.savefig(f"plots/fig5-deletion_homology.pdf", dpi=300,  bbox_inches='tight')
plt.show()

In [ ]:
## check fold-difference in exp vs ob for  >6bp 
conditions = [
    (hom_df['length'] <= 6),
    (hom_df['length'] > 6),
]
values = ['0-6', '7+']
hom_df['length_group2'] = np.select(conditions, values, default='-')

prop_df = hom_df.groupby('group')['length_group2'].value_counts(normalize=True).rename('proportion').reset_index()
prop_df 

In [ ]:
hom_df.dropna()

In [ ]:
## use wilcoxon-rank sum test to check sig diff bw expected and observed distributions of microhomology
from scipy.stats import mannwhitneyu

hom_df_nona = hom_df.dropna()

real = hom_df_nona[hom_df_nona.group == "Observed"]["length"]
offset = hom_df_nona[hom_df_nona.group == "Expected"]["length"]

stat, p = mannwhitneyu(real, offset, alternative="greater")
print(p)


In [ ]:
## generate annotated linear diagram for mtDNA
from Bio import SeqIO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches  # <-- missing import

record = SeqIO.read("jn_resources/mito.gb", "genbank")

fig, ax = plt.subplots(figsize=(12, 2.5))

# color scheme
colors = {
    "CDS": "#d73027",
    "rRNA": "#1a9850",
    "tRNA": "#4575b4",
    "control": "#984ea3"
}

for feature in record.features:
    start = int(feature.location.start)
    end = int(feature.location.end)
    width = end - start

    if feature.type == "CDS":
        color = colors["CDS"]
    elif feature.type == "rRNA":
        color = colors["rRNA"]
    elif feature.type == "tRNA":
        color = colors["tRNA"]
    elif feature.type in ["D-loop", "control_region"]:
        color = colors["control"]
    else:
        continue

    rect = patches.Rectangle((start, 0), width, 1, linewidth=0, facecolor=color)
    ax.add_patch(rect)

# y position just above the track
y_top = 1.05
OH = 407
OL = 5747
# draw vertical ticks
ax.plot([OH, OH], [0, y_top], color="black", linewidth=1, linestyle='dotted')
ax.plot([OL, OL], [0, y_top], color="black", linewidth=1, linestyle='dotted')

# add labels
ax.text(OH, y_top + 0.02, "OH", ha="center", va="bottom", fontsize=8)
ax.text(OL, y_top + 0.02, "OL", ha="center", va="bottom", fontsize=8)

## common deletion
cd_start = 8470
cd_end = 13447

ax.plot([cd_start, cd_end], [1.2, 1.2], color="black", linewidth=4)
ax.text((cd_start + cd_end)/2, 1.24, "Common deletion", ha="center", va="bottom", fontsize=8)

# formatting
ax.set_xlim(0, len(record.seq))
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_xlabel("Position (bp)")

legend_patches = [
    mpatches.Patch(color=colors["CDS"], label="Protein coding"),
    mpatches.Patch(color=colors["rRNA"], label="rRNA"),
    mpatches.Patch(color=colors["tRNA"], label="tRNA"),
    mpatches.Patch(color=colors["control"], label="Control region")
]

ax.legend(
    handles=legend_patches,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.2),  
    ncol=4,
    frameon=False,              
    fontsize=10,
    handlelength=1
)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout(rect=[0, 0.1, 1, 1])
ax.set_ylim(-0.3, 1.2)

plt.savefig("plots/fig5-mt_linear.pdf", dpi=300, bbox_inches="tight")
plt.show()